In [2]:
import bokeh.plotting as bpl
import cv2
import logging
import matplotlib.pyplot as plt
import numpy as np
import datetime
import fnmatch
import os

from cnmf_models import cnmf_2d


try:
    cv2.setNumThreads(0)
except():
    pass

try:
    if __IPYTHON__:
        # this is used for debugging purposes only. allows to reload classes
        # when changed
        get_ipython().magic('load_ext autoreload')
        get_ipython().magic('autoreload 2')
except NameError:
    pass

import caiman as cm
from caiman.motion_correction import MotionCorrect
from caiman.source_extraction.cnmf import cnmf as cnmf
from caiman.source_extraction.cnmf import params as params
from caiman.utils.utils import download_demo
from caiman.utils.visualization import plot_contours, nb_view_patches, nb_plot_contour
bpl.output_notebook()

logging.basicConfig(format=
                          "%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s] [%(process)d] %(message)s",
                    # filename="/tmp/caiman.log",
                    #level=logging.WARNING
                    )

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/var/folders/l9/cx2p5jb916q6gnl7zy00tcg00000gn/T/ipykernel_38212/3944797617.py:22: DeprecationWarning: `magic(...)` is deprecated since IPython 0.13 (warning added in 8.1), use run_line_magic(magic_name, parameter_s).
  get_ipython().magic('load_ext autoreload')
/var/folders/l9/cx2p5jb916q6gnl7zy00tcg00000gn/T/ipykernel_38212/3944797617.py:23: DeprecationWarning: `magic(...)` is deprecated since IPython 0.13 (warning added in 8.1), use run_line_magic(magic_name, parameter_s).
  get_ipython().magic('autoreload 2')


Loading BokehJS ...

In [ ]:
fname_load = ["/Users/wienecke/Documents/ambrose/leprechaunMat/20230627-2_D05_syt7f_018_syt7f/20230627-2_D05_syt7f_018_syt7f_181750_trial_002_00001.tif"]
fname = ["/Users/wienecke/Documents/ambrose/leprechaunMat/20230627-2_D05_syt7f_018_syt7f/20230627_2_regtmp.tif"]
fname_load = "/Users/wienecke/Documents/ambrose/leprechaunMat/20230627-2_D05_syt7f_018_syt7f/20230627_2_hires_registered_.tif"
fname = ["/Users/wienecke/Documents/ambrose/leprechaunMat/20230627-2_D05_syt7f_018_syt7f/20230627_2_hires_new_.tif"]
# Y = cm.load(fnames[0])
# Y = Y.reshape(80, 164, 140, 256)
# Y = Y[:,:-51,:,:] #crop flyback frames
# Y[20,:,:,:].play(magnification=2) 


if 'dview' in locals():
    cm.stop_server(dview=dview)
c, dview, n_processes = cm.cluster.setup_cluster(
    backend='local', n_processes=None, single_thread=False)

In [ ]:
only_init = False

sly = slice(0, 51, 1)
slx = slice(40, 181, 1) 
indices = [slx, sly]
indices = [slice(None), slice(None)]

p = 0                   # order of the autoregressive system - 0 from carl's code
merge_thresh = 0.9
gSig = [4, 4]  # gSig = [3,3]            # radius (half-size) of average neurons (in pixels)
nb = 2                  # temporal global background components - TUNE

do_patches = False      # flag for processing in patches or not - turn on or off - Not used in Matlab
if do_patches:          # PROCESS IN PATCHES AND THEN COMBINE
    rf = 60             # half size of each patch
    stride_cnmf = 40          # overlap between patches
    p_patch = p
    nb_patch = nb
    k = 3              # number of components in each patch
    indices = [slice(None), slice(None)]
else:                   # PROCESS THE WHOLE FOV AT ONCE
    rf = None           # setting these parameters to None
    stride_cnmf = None       # will run CNMF on the whole FOV
    p_patch = p
    nb_patch = nb
    k = 100              # number of neurons expected (in the whole FOV) - 40 from Carl's Code, seems to be too many

###
fr = 5.08 #0.6193  #9.8465 frame period so 1000 / (9.8465 *(113+51)) # approximate frame rate of data - CONFIRMED FPS
decay_time = .4         # length of transient - CONFIRMED APPROPRIATE FOR OUR INDICATOR GCaMP6f
dxy = [1.33155792277, 1.33155792277] #for .751 um pixels # pixels per micron 

tsub = 1                # temporal downsampling
ssub = 1               # spatial downsampling

method_init = 'greedy_roi' #'graph_nmf' #'greedy_roi' #python Caiman defaults to greedy_roi, carl's code uses sparse_nmf, but sparse_nmf runs MUCH slower
sigma_smooth_snmf = (0.5, 0.5, 0.5)
perc_baseline_snmf = 50

se = np.ones((3,)*len(gSig), dtype=np.uint8)  #se = np.ones((3,3,1), dtype=np.uint8)
update_background_components = True   #use this??

fudge_factor = 0.96        # (default is 0.96; Carl's value = 1) -- bias correction factor for discrete time constants
ITER = 5                # (default is 2; Carl's value=5) -- block coordinate descent iterations
bas_nonneg = False #True

rolling_sum = True

min_SNR = 0      # accept components with that peak-SNR or higher (if above this, acept)
SNR_lowest = 0         # minimum SNR for accepted components (if below this, reject)
rval_lowest = 0  # 0.6  # space correlation threshold (if above this, accept)
rval_thr = 0  # 0.6  # space correlation threshold (if above this, accept)
use_cnn = False      # use the CNN classifier affects if 2 below params are used
min_cnn_thr = 0  # if cnn classifier predicts below this value, reject
cnn_lowest = 0.1   # neurons with cnn probability lower than this value are rejected


In [ ]:
opts_dict = {'fnames': fname,
            'fr': fr,
            'p': p,
            'nb': nb,
            'merge_thr': merge_thresh,
            'rf': rf,
            'K': k, 
            'gSig': gSig,
            'stride': stride_cnmf,
            'method_init': method_init,
            'sigma_smooth_snmf': sigma_smooth_snmf,
            'perc_baseline_snmf': perc_baseline_snmf,
            #'dims': dims_new,
            'dxy': dxy,
            'decay_time': decay_time,
            'bas_nonneg': bas_nonneg,
            'p_patch': p_patch,
            'nb_patch': nb_patch,
            'se': se, 
            'rolling_sum': rolling_sum,
            'only_init': only_init,
            'ssub': ssub,
            'tsub': tsub,
            'SNR_lowest': SNR_lowest, 
            'min_SNR': min_SNR,
            'rval_thr': rval_thr,
            'use_cnn': use_cnn,
            'min_cnn_thr': min_cnn_thr,
            'cnn_lowest': cnn_lowest, 
            'ITER': ITER, 
            'fudge_factor': fudge_factor, 
            'cnn_lowest': cnn_lowest, 
            'update_background_components': update_background_components}

opts = params.CNMFParams(params_dict=opts_dict)

In [ ]:

recordingIDs = ['20230627-2']
data_path_prefix = '/Users/wienecke/Documents/ambrose/leprechaunMat/'

for recordingID in recordingIDs:
  
  tmpdate = datetime.datetime.now().strftime("%Y%m%dT%H%M%S") #create extraction ID, one for each recordingID

  path_date_stack = data_path_prefix + recordingID + '_D05_syt7f_018_syt7f/'

  path_to_stacks = []

  stack_filename_template = recordingID + '*.tif'

  for f in os.listdir(path_date_stack):
    if fnmatch.fnmatch(f,stack_filename_template):
      tmp = path_date_stack + f
      path_to_stacks.append(tmp)

  for f in path_to_stacks:

    cnm_object_out, results = cnmf_2d(path_to_stack = f, #from the pairs of files above, could be swapped out for string directly
                                      opts = opts) #params object globally initialized earlier

    #visualize results
    cnm_object_out.estimates.dims = [128,256]
    print(f"ROIs extracted from {results['stack_name']}")

    fname_save = f[:(len(f)-4)] + '_caimanrois_.mat'

    filename_save_h5 = fname_save + tmpdate + '_.h5'
    filename_save_mat = fname_save + tmpdate + '_.mat'

    #mdict['spatial'] = np.transpose(cnm2.estimates.A.A, axes = (1,0)) #A.A is regular array version not sparse?
    dims_roistack = [dims[1], dims[0], dims[2], results.A.shape[-1]]
    #roistack = np.reshape(cnm2.estimates.A.toarray(), dims_roistack, order='F') #the roi stack, binary version below

    masks = np.zeros(dims_roistack,  dtype=bool)
    M = results.A > 0
    for i in range(M.shape[1]):
        masks[:,:,:,i] = np.reshape(M[:,i].toarray(), dims, order='F').transpose(1, 0, 2) #binary mask version of roi stack  

    mdict = {}
    mdict['roimasks'] = masks
    mdict['C'] = results.C
    mdict['YrA'] = results.YrA
    mdict['S'] = results.S
    if results.F_dff is None:
        mdict['dff'] = 0
    else:
        mdict['dff'] = results.F_dff
    sio.savemat(filename_save_mat,mdict)

    # with h5py.File(filename_save_h5, "w") as result_file:
    #     result_file.create_dataset('stack_name', data=results['stack_name'])
    #     result_file.create_dataset('extractionID', data=tmpdate)
    #     result_file.create_dataset('temporal', data=results['temporal'])
    #     result_file.create_dataset('spatial', data=results['spatial'])

    cnm_object_out.estimates.nb_view_components(denoised_color = 'red')



In [ ]:
cnm1 = cnmf.CNMF(n_processes, params=opts, dview=dview)

cnm1.fit_file(motion_correct=False)

In [ ]:
dims = (256, 140, 113)
cnm1.estimates.nb_view_components_3d(image_type='mean', dims=dims, axis=2)

In [ ]:
cnm2.estimates.nb_view_components_3d(image_type='corr', dims=dims, Yr=Yr, denoised_color='red', max_projection=True);

In [ ]:
%%capture
cnm.params.set('patch', {'only_init': False})
cnm.params.set('data', {'use_cnn': False})

cnm2 = cnm.refit(images)